# Series 2.4 — Conversation Summarization

**Why AI Fails? — Engineering Lab**

---

> Conversations grow forever. Memory shouldn't.

**Scenario:** A 175-message enterprise AI coding assistant thread — four summarization strategies, one benchmark question.

**Core lesson:** Conversation summarization is **memory management** — remember the **right information**, not every message.

## 1. The Problem

| Without summarization | With summarization |
|-----------------------|--------------------|
| All 175 messages in every prompt | Summary + recent messages only |
| Prompt size grows linearly with chat length | Prompt size stays bounded |
| High cost, high latency | Much lower cost, faster responses |
| Model sees noise (greetings, thanks) | Model sees facts and decisions |

### Why this matters in production

- Support bots and coding assistants accumulate **unbounded history**
- Replaying full transcripts on every turn wastes tokens on irrelevant chatter
- The model needs **decisions, preferences, and pending tasks** — not "good morning"

**Expected dry-run insight:**

```
FULL          → highest Memory Score, highest prompt tokens
ROLLING       → good memory, much lower cost
HIERARCHICAL  → scales to very long threads
SEMANTIC      → highest information density per token
```

## 2. What is Conversation Summarization?

**Conversation summarization** compresses chat history into a **compact memory representation** while preserving facts the assistant must remember across turns.

It is not about shortening the user's latest question. It is about **not replaying the entire conversation** on every API call.

### Definition

```
Conversation summarization = full history → memory + recent window → prompt → LLM
                               (runs BEFORE each new turn)
```

### Four strategies in this lab

| Strategy | What enters the prompt |
|----------|------------------------|
| **Full** | Entire 175-message history (baseline) |
| **Rolling** | Evolving summary + latest 10 messages |
| **Hierarchical** | Block summaries → master summary + latest 10 |
| **Semantic** | Structured facts only + latest 5 messages |

### What summarization is NOT

| Technique | Difference |
|-----------|------------|
| **Context pruning** (Series 2.1) | Pruning filters external evidence (logs); summarization compresses **chat** |
| **Long-term memory** (Series 2.5) | Summarization is **session-scoped**; long-term memory persists **across sessions** |
| **RAG chunking** (Series 2.3) | RAG chunks **documents**; summarization chunks **conversation turns** |

> **Enterprise principle:** Summarize deterministically where possible — reproducible dry-runs, testable memory scores.

## 3. Repository Layout

```
why-ai-fails/
├── common/                        ← Gemini client, token math
└── series-2.4/
    ├── app.py                     ← CLI + benchmark runner
    ├── conversation_dataset.py    ← ~175 synthetic messages
    ├── conversation_loader.py     ← Load conversation into memory
    ├── summarizer.py              ← Four summarization strategies
    ├── memory.py                  ← Structured semantic memory
    ├── evaluator.py               ← Memory Score + retention
    ├── prompts.py                 ← Prompt templates
    ├── benchmark.py               ← Side-by-side comparison
    ├── README.md
    └── Series_2.4_Conversation_Summarization.ipynb   ← This notebook
```

## 4. The Summarization Pipeline (`summarizer.py`)

**Without summarization:**
```
175 Messages → Prompt Builder → Gemini
```

**With summarization:**
```
175 Messages → Summarizer → Memory + Latest Messages → Prompt → Gemini
```

| Strategy | Window | Compression method |
|----------|--------|--------------------|
| Rolling | Latest 10 verbatim | Bullet summary of older messages |
| Hierarchical | Latest 10 verbatim | 20-msg blocks → master summary |
| Semantic | Latest 5 verbatim | Extract structured facts; drop noise |

Summaries are built **locally** (no LLM) so `--dry-run` works without an API key.

## 5. Three Layers of Conversation Memory

### Layer 1 — Preserve what matters (Memory Score)

Benchmark questions test whether summarization retained:
- Python preference, project name, architectural decisions, pending tasks

**Memory Score** = remembered facts / expected facts

---

### Layer 2 — Compress before the prompt

Filter noise phrases (`thank you`, `good morning`, …) from summaries. Semantic strategy keeps **structured facts** only.

---

### Layer 3 — Measure tokens vs memory

| Metric | What it tells you |
|--------|-------------------|
| **Prompt tokens** | Cost of replaying history |
| **Summary size** | Compressed memory footprint |
| **Memory Score** | Did we lose critical facts? |
| **Context retention** | Score relative to full conversation |

| Mode | Flag | API key? |
|------|------|----------|
| **Dry-run** | `--dry-run` | No — **$0** |
| **Live** | (none) | Yes — 4 Gemini calls (one per strategy) |

## 6. Execution Flow

```
Parse CLI args (--strategy, --question-id)
    │
    └─ Load 175-message conversation
            │
            For each strategy (full / rolling / hierarchical / semantic):
                build_summary() → memory + recent messages
                build_prompt() → token estimate or Gemini
                compute Memory Score
            │
            └─ print_benchmark()
```

## 7. How to Run

From the **repo root**:

```bash
pip install -r requirements.txt
cp .env.example .env   # optional
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python series-2.4/app.py --dry-run` | All four strategies | No |
| `python series-2.4/app.py --strategy semantic --dry-run` | Single strategy | No |
| `python series-2.4/app.py --question-id q5 --dry-run` | Different question | No |
| `python series-2.4/app.py` | Live Gemini | Yes |

In [ ]:
# Live demo cell — run the dry-run benchmark ($0, no API key needed)
# Execute this cell during your presentation

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "demo.py").exists() and (ROOT.parent / "demo.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.4/app.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")


## 8. Key Code Snippets

### Strategy windows (`summarizer.py`)

```python
LATEST_ROLLING = 10
LATEST_HIERARCHICAL = 10
LATEST_SEMANTIC = 5
HIERARCHICAL_BLOCK = 20
```

### Noise filtering

```python
NOISE_PHRASES = ("thank you", "thanks", "good morning", "got it", ...)

def _is_low_value(content: str) -> bool:
    # Semantic strategy discards greetings and acknowledgments
    ...
```

## 9. Where Series 2.4 Fits

| Lab | Topic | Role in the stack |
|-----|-------|-------------------|
| 2.1–2.3 | Prune, cache, chunk | Shrink and retrieve external knowledge |
| **2.4** | **Conversation Summarization** | **Session memory** — bounded chat context |
| 2.5 | Long-Term Memory | Persistent facts across 500 conversations |
| 2.6 | Memory Retrieval | Find the right memory from 100k records |
| 2.7 | Model Routing | Route to the right model tier |

---

## Takeaway

> **The objective is not to remember every message. The objective is to remember the right information.**

**Next lab:** [Series 2.5 — Long-Term Memory](../series-2.5/) — compress durable user knowledge across hundreds of conversations.